In [1]:
from pathlib import Path
from datetime import datetime
from typing import List
from types import SimpleNamespace
import pickle
import numpy as np
import torch
from pymatgen.core import Structure, Lattice, Element


from chggen.pl_data.dataset import CHGNetDataset
from chggen.pl_modules.model_volume import CHGGen
from chggen.common.data_utils import get_scaler_from_data_list, get_scaler
import os

import pytorch_lightning as pl

def mkdir(path: str):
    folder = os.path.exists(path)
    if not folder:
        os.makedirs(path)
    else:
        print("Folder exists")
    return path

/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

device = torch.device('cuda')


with open('./test_models/lattice_scaler_perov', 'rb') as fp:
    lattice_scaler = pickle.load(fp)


chggen = CHGGen.load_from_checkpoint('./test_models/perov/epoch=2.ckpt')
chggen.to(device = device)

chggen.lattice_scaler = lattice_scaler




dataset = CHGNetDataset(
    path='./data/perov_5/test_zpc.csv',
    name = 'A_good_name',
    prop_list = ['heat_all'],
)

# lattice_scaler = get_scaler(dataset= dataset)


CHGNet initialized with 412,525 parameters
CHGNet v0.3.0 initialized with 412,525 parameters


/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/torch/overrides.py:110: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  torch.has_cuda,
/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/torch/overrides.py:111: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  torch.has_cudnn,
/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/torch/overrides.py:117: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  torch.has_mps,
/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/torch/overrides.py:118: UserWarning: 'has_mkldnn' is deprecated, please use 'torch.backends.mkldnn.is_available()'
  torch.has_mkldnn,
/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use 

In [3]:
graph_list = [dataset[ii].crys_graph.to(chggen.device) for ii in range(len(dataset))]

In [4]:
_, _, z_reconst =  chggen.encode(graph_list)

z_reconst = z_reconst[:2]

In [5]:
# chggen.reparameterize(crystal_features, )

In [6]:
torch.diag(dataset[5].lattices)

tensor([4.0788])

In [7]:

# langevin dynamics
ld_kwargs = SimpleNamespace(n_step_each = 10,
                            step_lr = 1e-3,
                            min_sigma = 0,
                            save_traj = False,
                            disable_bar = False,
                            compute_force = True,
                            beta_c = 0, # property update rate
                            beta_f = 0, # atomic force update rate
                            )


num_structures = len(z_reconst)
# z = torch.rand(num_structures, 64, requires_grad= True, device = device)
results = chggen.diffusion_quench_guidance(z = z_reconst, 
                                           prop_guidance = torch.tensor(-0.05, device= device), 
                                           box_lengths = [4.1, 4.1, 4.1],
                                           box_angles = [90, 90, 90],
                                           # gt_num_atoms = torch.ones(num_structures, device = device, dtype = torch.int64) * 5, # 
                                           # box_lengths = 4.1*1,
                                           # box_angles = 90,
                                           change_type = True, 
                                           ld_kwargs= ld_kwargs)


/home/zhongpc/chggen/chggen/pl_modules/model_volume.py:318: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  num_atoms = torch.tensor(num_atoms, dtype= torch.int64,  device= z.device)
/home/zhongpc/chggen/chggen/common/data_utils.py:637: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float)


Atom volume tensor([[15.1147],
        [10.4038]], device='cuda:0', grad_fn=<AddmmBackward0>)
Init atom type:  tensor([23,  7, 40,  8,  8,  7,  8,  9, 28,  8, 14,  7], device='cuda:0')


  0%|                                                                                                         | 0/50 [00:00<?, ?it/s]

tensor(10., device='cuda:0')


  2%|█▉                                                                                               | 1/50 [00:02<02:15,  2.77s/it]

...

 96%|████████████████████████████████████████████████████████████████████████████████████████████▏   | 48/50 [01:14<00:03,  1.62s/it]

tensor(0.0012, device='cuda:0')


 98%|██████████████████████████████████████████████████████████████████████████████████████████████  | 49/50 [01:16<00:01,  1.58s/it]

tensor(0.0010, device='cuda:0')


100%|████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [01:17<00:00,  1.56s/it]


In [8]:
# STOP

In [9]:
from chgnet.model import StructOptimizer

relaxer = StructOptimizer(model = chggen.encoder.model)


CHGNet will run on cuda:0


In [10]:
mkdir('./test_models/reconst_structures/')

Folder exists


'./test_models/reconst_structures/'

In [11]:

# save the results from langevin dynamics
lengths = results['lengths']
angles= results['angles']
num_atoms = results['num_atoms']
frac_coords = results['frac_coords']
atom_types = results['atom_types']

batch = torch.arange(len(num_atoms), device = device)
batch = batch.repeat_interleave(num_atoms)
print(num_atoms)
for ii in range(len(num_atoms)):
    indices = torch.where(batch == ii)[0]
    # print(ii, indices, )

    crys_graph = dataset[ii].crys_graph
    # print("composition", crys_graph.composition)
    print("num atoms: ", len(indices))

    if len(indices) == 0:
        continue
    
    
    Latt = Lattice.from_parameters(a = lengths.cpu().detach().numpy()[ii,0], 
                                   b = lengths.cpu().detach().numpy()[ii,1], 
                                   c = lengths.cpu().detach().numpy()[ii,2],
                                   alpha= angles.cpu().detach().numpy()[ii, 0], 
                                   beta = angles.cpu().detach().numpy()[ii,1], 
                                   gamma= angles.cpu().detach().numpy()[ii, 2])
                                   
    frac_ = frac_coords[indices]
    type_ = atom_types[indices]
    species_ = [Element.from_Z(ele_Z) for ele_Z in type_]
    
    s_gen = Structure(lattice= Latt , species= species_, coords= frac_.cpu().detach().numpy(),
                      to_unit_cell=False,coords_are_cartesian=False);
    s_gen.sort()
    print("previou compo: ", crys_graph.composition)
    print("reconst compo: ", s_gen.composition)
    s_gen.to(filename= './test_models/reconst_structures/reconst_' + str(ii) + '.cif')
print("Done")

tensor([5, 7], device='cuda:0')
num atoms:  5
previou compo:  Ti1 Os1 N1 O1 F1
reconst compo:  Zr1 V1 N1 O2
num atoms:  7
previou compo:  Cu1 As1 N3
reconst compo:  Al1 Si1 Ni1 N2 O1 F1
Done


In [12]:
result = relaxer.relax(s_gen, fmax= 0.5, steps = 100)
print("CHGNet relaxed structure", result["final_structure"])
print("relaxed total energy in eV:", result['trajectory'].energies[-1])

      Step     Time          Energy         fmax
*Force-consistent energies used in optimization.
FIRE:    0 11:20:42       -7.774500*      93.3996
FIRE:    1 11:20:42      -17.648628*     139.9084
FIRE:    2 11:20:42      -25.010038*     115.6557
FIRE:    3 11:20:42      -34.033129*      93.1838
FIRE:    4 11:20:42      -37.022403*      19.8830
FIRE:    5 11:20:42      -37.190023*      15.3076
FIRE:    6 11:20:42      -37.394741*      14.7848
FIRE:    7 11:20:42      -37.660959*      12.9635
FIRE:    8 11:20:42      -37.904629*      15.2383
FIRE:    9 11:20:42      -38.159819*      21.0616
FIRE:   10 11:20:42      -38.612269*      12.6600
FIRE:   11 11:20:42      -39.047202*      14.0771
FIRE:   12 11:20:42      -39.582992*      15.6941
FIRE:   13 11:20:42      -40.227446*      13.7332
FIRE:   14 11:20:42      -40.958120*      13.9555
FIRE:   15 11:20:42      -41.719870*      21.2009
FIRE:   16 11:20:42      -42.154770*      17.1976
FIRE:   17 11:20:42      -42.147827*       9.7049
FI

In [13]:
result['final_structure'].to(filename='./test_models/reconst_structures/chgnet.cif')

"# generated using pymatgen\ndata_AlSiNiN2OF\n_symmetry_space_group_name_H-M   'P 1'\n_cell_length_a   3.59896155\n_cell_length_b   4.03288509\n_cell_length_c   4.86324843\n_cell_angle_alpha   77.12263526\n_cell_angle_beta   86.48230970\n_cell_angle_gamma   113.63015750\n_symmetry_Int_Tables_number   1\n_chemical_formula_structural   AlSiNiN2OF\n_chemical_formula_sum   'Al1 Si1 Ni1 N2 O1 F1'\n_cell_volume   62.13741391\n_cell_formula_units_Z   1\nloop_\n _symmetry_equiv_pos_site_id\n _symmetry_equiv_pos_as_xyz\n  1  'x, y, z'\nloop_\n _atom_site_type_symbol\n _atom_site_label\n _atom_site_symmetry_multiplicity\n _atom_site_fract_x\n _atom_site_fract_y\n _atom_site_fract_z\n _atom_site_occupancy\n  Al  Al0  1  0.98927490  0.02592590  0.94201098  1\n  Si  Si1  1  0.05584727  0.04369808  -0.03645512  1\n  Ni  Ni2  1  1.02341975  0.01420670  0.00178717  1\n  N  N3  1  0.92334355  0.10234025  0.30420315  1\n  N  N4  1  0.63308513  0.53712718  0.08841163  1\n  O  O5  1  0.52491375  0.2283752

In [14]:
sigma_begin = 10
sigma_end = 0.1
num_noise_level = 50

sigmas = torch.tensor(np.exp(np.linspace(
            np.log(sigma_begin),
            np.log(sigma_end),
            num_noise_level)), dtype=torch.float32)

In [15]:
sigmas

tensor([10.0000,  9.1030,  8.2864,  7.5431,  6.8665,  6.2506,  5.6899,  5.1795,
         4.7149,  4.2919,  3.9069,  3.5565,  3.2375,  2.9471,  2.6827,  2.4421,
         2.2230,  2.0236,  1.8421,  1.6768,  1.5264,  1.3895,  1.2649,  1.1514,
         1.0481,  0.9541,  0.8685,  0.7906,  0.7197,  0.6551,  0.5964,  0.5429,
         0.4942,  0.4498,  0.4095,  0.3728,  0.3393,  0.3089,  0.2812,  0.2560,
         0.2330,  0.2121,  0.1931,  0.1758,  0.1600,  0.1456,  0.1326,  0.1207,
         0.1099,  0.1000])